In [1]:
import torch
import torchshow as ts

In [2]:
def intersection_over_union(boxes_preds, boxes_labels, is_yolo:bool=True):
    """
    Calculates Intersection over Union (IoU) with support for broadcasting.
    
    Args:
        boxes_preds (Tensor): Predicted boxes, shape [..., num_boxes, 4]
        boxes_labels (Tensor): Target boxes, shape [..., 1, 4]
        box_format (str): 'midpoint' (x,y,w,h) or 'corners' (x1,y1,x2,y2)
        
    Returns:
        Tensor: IoU for all pairs, shape [..., num_boxes]
    """
    if is_yolo:
        pred_w = boxes_preds[..., 2:3].clamp(min=1e-6)
        pred_h = boxes_preds[..., 3:4].clamp(min=1e-6)

        target_w = boxes_labels[..., 2:3].clamp(min=1e-6)
        target_h = boxes_labels[..., 3:4].clamp(min=1e-6)

        box1_x1 = boxes_preds[..., 0:1] - pred_w / 2
        box1_y1 = boxes_preds[..., 1:2] - pred_h / 2
        box1_x2 = boxes_preds[..., 0:1] + pred_w / 2
        box1_y2 = boxes_preds[..., 1:2] + pred_h / 2

        box2_x1 = boxes_labels[..., 0:1] - target_w / 2
        box2_y1 = boxes_labels[..., 1:2] - target_h / 2
        box2_x2 = boxes_labels[..., 0:1] + target_w / 2
        box2_y2 = boxes_labels[..., 1:2] + target_h / 2

    else:
        box1_x1 = boxes_preds[..., 0:1]
        box1_y1 = boxes_preds[..., 1:2]
        box1_x2 = boxes_preds[..., 2:3]
        box1_y2 = boxes_preds[..., 3:4]
        
        box2_x1 = boxes_labels[..., 0:1]
        box2_y1 = boxes_labels[..., 1:2]
        box2_x2 = boxes_labels[..., 2:3]
        box2_y2 = boxes_labels[..., 3:4]

    # 1. Calculate intersection coordinates using element-wise max/min
    x1 = torch.max(box1_x1, box2_x1)
    y1 = torch.max(box1_y1, box2_y1)
    x2 = torch.min(box1_x2, box2_x2)
    y2 = torch.min(box1_y2, box2_y2)

    # 2. Compute intersection area (.clamp(0) handles cases with zero overlap)
    intersection = (x2 - x1).clamp(min=0) * (y2 - y1).clamp(min=0)

    # 3. Compute individual box areas
    box1_area = abs((box1_x2 - box1_x1) * (box1_y2 - box1_y1))
    box2_area = abs((box2_x2 - box2_x1) * (box2_y2 - box2_y1))

    # 4. Calculate union area (Area A + Area B - Intersection)
    union = box1_area + box2_area - intersection

    # 5. Return IoU score (1e-6 prevents dividing by absolute zero)
    return (intersection / (union + 1e-6)).squeeze(-1)

In [4]:
S = 7
B = 2
C = 1
predictions = torch.rand(size=(1, S*S*(5*B + C)))
target = torch.rand(size=(1, S, S, C + 5))
target[..., 0] = (target[..., 0] > 0.5).to(target.dtype)
target[..., 1] = (target[..., 1] > 0.5).to(target.dtype)
predictions, target

(tensor([[0.5589, 0.1520, 0.6541, 0.3514, 0.8407, 0.6570, 0.0367, 0.1921, 0.5326,
          0.0325, 0.8083, 0.6105, 0.1905, 0.1414, 0.8325, 0.3608, 0.5511, 0.1246,
          0.6236, 0.7727, 0.0073, 0.6839, 0.9173, 0.0227, 0.7399, 0.2362, 0.3558,
          0.6148, 0.0095, 0.3552, 0.9872, 0.6250, 0.4627, 0.8776, 0.3479, 0.6332,
          0.0756, 0.1293, 0.6516, 0.2036, 0.5505, 0.0133, 0.0525, 0.1844, 0.1406,
          0.8045, 0.7185, 0.2667, 0.8165, 0.0455, 0.8874, 0.3863, 0.7171, 0.2908,
          0.8823, 0.5164, 0.2981, 0.6679, 0.9623, 0.6905, 0.3225, 0.7910, 0.9115,
          0.4585, 0.2897, 0.4188, 0.5692, 0.6343, 0.3044, 0.4196, 0.4759, 0.8756,
          0.5678, 0.2209, 0.0677, 0.7729, 0.0473, 0.3340, 0.0547, 0.1812, 0.8187,
          0.7833, 0.1981, 0.0083, 0.3048, 0.5813, 0.2267, 0.4719, 0.6356, 0.0940,
          0.5383, 0.7689, 0.9050, 0.4067, 0.2629, 0.0351, 0.5930, 0.4444, 0.6723,
          0.3660, 0.2436, 0.6876, 0.8464, 0.8452, 0.5387, 0.2836, 0.4257, 0.2632,
          0.4987

In [5]:
predictions = predictions.reshape(-1, S, S, B * 5 + C)
start_index = C + B
all_pred_boxes = predictions[..., start_index:(start_index + B*4)].reshape(*predictions.shape[:-1], B, 4)
target_box = target[..., (C+1):(C+5)].unsqueeze(-2)
ious = intersection_over_union(all_pred_boxes, target_box)
ious_max = torch.argmax(ious, dim=-1)
best_box_expanded = ious_max.unsqueeze(-1).unsqueeze(-1).expand(-1, -1, -1, 1, 4)

In [12]:
best_box_expanded

tensor([[[[[0, 0, 0, 0]],

          [[0, 0, 0, 0]],

          [[0, 0, 0, 0]],

          [[0, 0, 0, 0]],

          [[1, 1, 1, 1]],

          [[1, 1, 1, 1]],

          [[0, 0, 0, 0]]],


         [[[1, 1, 1, 1]],

          [[0, 0, 0, 0]],

          [[1, 1, 1, 1]],

          [[1, 1, 1, 1]],

          [[0, 0, 0, 0]],

          [[1, 1, 1, 1]],

          [[0, 0, 0, 0]]],


         [[[1, 1, 1, 1]],

          [[1, 1, 1, 1]],

          [[1, 1, 1, 1]],

          [[0, 0, 0, 0]],

          [[0, 0, 0, 0]],

          [[0, 0, 0, 0]],

          [[1, 1, 1, 1]]],


         [[[0, 0, 0, 0]],

          [[1, 1, 1, 1]],

          [[0, 0, 0, 0]],

          [[0, 0, 0, 0]],

          [[0, 0, 0, 0]],

          [[0, 0, 0, 0]],

          [[0, 0, 0, 0]]],


         [[[0, 0, 0, 0]],

          [[1, 1, 1, 1]],

          [[0, 0, 0, 0]],

          [[1, 1, 1, 1]],

          [[0, 0, 0, 0]],

          [[0, 0, 0, 0]],

          [[0, 0, 0, 0]]],


         [[[0, 0, 0, 0]],

          [[1, 1, 

In [13]:
exists_prob = target[..., C]
exists_box = exists_prob.unsqueeze(-1).unsqueeze(-1)
best_pred_box = torch.gather(all_pred_boxes, dim=-2, index=best_box_expanded)

In [ ]:
exists_box

tensor([[[1., 0., 1., 1., 0., 1., 0.],
         [0., 1., 0., 1., 0., 1., 1.],
         [0., 1., 1., 0., 1., 1., 0.],
         [0., 1., 0., 0., 1., 1., 1.],
         [1., 0., 1., 0., 1., 0., 0.],
         [0., 0., 0., 1., 1., 0., 1.],
         [1., 0., 0., 1., 1., 0., 0.]]])

In [ ]:
box_predictions = exists_box*best_pred_box
box_targets = exists_box*target_box

In [20]:
box_predictions.shape, exists_box.shape

(torch.Size([1, 7, 7, 1, 4]), torch.Size([1, 7, 7, 1, 1]))

In [21]:
best_iou = torch.gather(
            ious,
            dim=-1,
            index=ious_max.unsqueeze(-1)
        ).detach()
best_iou

tensor([[[[0.0137],
          [0.0000],
          [0.0000],
          [0.1637],
          [0.3024],
          [0.0148],
          [0.3666]],

         [[0.1499],
          [0.0400],
          [0.2066],
          [0.2194],
          [0.0000],
          [0.0438],
          [0.0000]],

         [[0.1318],
          [0.4760],
          [0.0260],
          [0.0028],
          [0.3633],
          [0.5466],
          [0.2290]],

         [[0.1402],
          [0.0156],
          [0.4143],
          [0.0000],
          [0.2267],
          [0.0000],
          [0.0000]],

         [[0.0000],
          [0.0836],
          [0.0000],
          [0.0909],
          [0.0000],
          [0.0000],
          [0.0000]],

         [[0.0000],
          [0.0281],
          [0.0505],
          [0.2672],
          [0.0322],
          [0.1470],
          [0.3056]],

         [[0.2104],
          [0.2644],
          [0.1084],
          [0.0665],
          [0.1625],
          [0.0143],
          [0.0168]]]])

In [3]:
from pathlib import Path
import os

model_path = Path('../model')
epoch_list = list(model_path.glob('*.pth'))

In [ ]:
import torch
device = torch.accelerator.current_accelerator() if torch.accelerator.is_available() else 'cpu'

for model in epoch_list:
    data = torch.load(model, map_location=device)
    print(f"{model.stem} has loss {data['loss']}\n")

Epoch Epoch-78 has loss 0.9467575550079346

Epoch Epoch-86 has loss 0.921588659286499

Epoch Epoch-74 has loss 0.9364756941795349

Epoch Epoch-75 has loss 0.949755072593689

Epoch Epoch-88 has loss 0.9356716275215149

Epoch Epoch-82 has loss 0.9487631320953369

Epoch Epoch-90 has loss 0.9486351013183594

Epoch Epoch-84 has loss 0.9485058188438416

Epoch Epoch-76 has loss 0.9259016513824463

Epoch Epoch-70 has loss 0.9488301277160645

Epoch Epoch-77 has loss 0.9382728934288025

Epoch Epoch-79 has loss 0.950641393661499

Epoch Epoch-85 has loss 0.9529159665107727

Epoch Epoch-87 has loss 0.9317145347595215

Epoch Epoch-83 has loss 0.9287305474281311

Epoch Epoch-81 has loss 0.9475973844528198

Epoch Epoch-89 has loss 0.9370853304862976

Epoch Epoch-80 has loss 0.9744622111320496

